In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.1f}".format)

CARTELLA_GREZZI = Path("../dati_grezzi")
CARTELLA_PULITI = Path("../dati_puliti")
CARTELLA_PULITI.mkdir(exist_ok=True)

# Cerchiamo il file MEF nella cartella (così funziona anche se il nome cambia leggermente)
file_mef = sorted(CARTELLA_GREZZI.glob("Redditi_e_principali_variabili_IRPEF*comunale*.csv"))[0]
print("File trovato:", file_mef.name)

File trovato: Redditi_e_principali_variabili_IRPEF_su_base_comunale_CSV_2024.csv


In [2]:
mef = pd.read_csv(
    file_mef,
    sep=";",
    encoding="latin1",
    dtype={"Codice Istat Comune": str, "Codice Istat Regione": str},
    keep_default_na=False,
    na_values=[""],
)

# Il file termina con un ';' in più: pandas crea una colonna vuota senza nome, la togliamo
mef = mef.loc[:, mef.columns.str.strip() != ""]

print("Righe (comuni):", len(mef), "| Colonne:", mef.shape[1])
print("Anno d'imposta:", mef["Anno di imposta"].unique())
mef.iloc[:3, :8]  # prime 3 righe, prime 8 colonne

Righe (comuni): 7897 | Colonne: 52
Anno d'imposta: [2024]


,Anno di imposta,Codice catastale,Codice Istat Comune,Denominazione Comune,Sigla Provincia,Regione,Codice Istat Regione,Numero contribuenti
0,2024,A001,028001,ABANO TERME,PD,Veneto,05,15850
1,2024,A004,098001,ABBADIA CERRETO,LO,Lombardia,03,210
2,2024,A005,097001,ABBADIA LARIANA,LC,Lombardia,03,2506


In [3]:
PROVINCE = {
    "015": "Milano", "016": "Bergamo", "018": "Pavia",
    "058": "Roma", "059": "Latina", "060": "Frosinone",
    "061": "Caserta", "063": "Napoli", "065": "Salerno",
    "071": "Foggia", "072": "Bari", "075": "Lecce",
}

mef["codice_provincia"] = mef["Codice Istat Comune"].str[:3]
mef12 = mef[mef["codice_provincia"].isin(PROVINCE)].copy()

# Controllo: quanti comuni per provincia? Devono esserci tutte e 12, Napoli compresa
conteggio = mef12.groupby(["codice_provincia", "Sigla Provincia"]).size().rename("n_comuni")
print("Province trovate:", len(conteggio))
conteggio

Province trovate: 12


codice_provincia  Sigla Provincia
015               MI                 133
016               BG                 243
018               PV                 185
058               RM                 121
059               LT                  33
060               FR                  91
061               CE                 104
063               NA                  92
065               SA                 158
071               FG                  61
072               BA                  41
075               LE                  96
Name: n_comuni, dtype: int64

In [4]:
COL_REDDITO = "Reddito complessivo - Ammontare in euro"
COL_FREQ = "Reddito complessivo - Frequenza"
FASCE_BASSE = [
    "Reddito complessivo minore o uguale a zero euro - Frequenza",
    "Reddito complessivo da 0 a 10000 euro - Frequenza",
    "Reddito complessivo da 10000 a 15000 euro - Frequenza",
]

mef12["contribuenti_sotto_15k"] = mef12[FASCE_BASSE].fillna(0).sum(axis=1)

redditi = mef12.groupby("codice_provincia").agg(
    sigla=("Sigla Provincia", "first"),
    n_comuni=("Codice Istat Comune", "count"),
    contribuenti=("Numero contribuenti", "sum"),
    contribuenti_reddito=(COL_FREQ, "sum"),
    reddito_totale=(COL_REDDITO, "sum"),
    contribuenti_sotto_15k=("contribuenti_sotto_15k", "sum"),
).reset_index()

redditi["provincia"] = redditi["codice_provincia"].map(PROVINCE)
redditi["reddito_medio"] = (redditi["reddito_totale"] / redditi["contribuenti_reddito"]).round(0)
redditi["quota_sotto_15k"] = (redditi["contribuenti_sotto_15k"] / redditi["contribuenti_reddito"] * 100).round(1)
redditi["anno_imposta"] = 2024

redditi = redditi[["codice_provincia", "provincia", "sigla", "anno_imposta", "n_comuni",
                   "contribuenti", "reddito_totale", "reddito_medio", "quota_sotto_15k"]]
redditi.sort_values("reddito_medio", ascending=False)

,codice_provincia,provincia,sigla,anno_imposta,n_comuni,contribuenti,reddito_totale,reddito_medio,quota_sotto_15k
0,015,Milano,MI,2024,133,2486066,84037688556,"34,803.0",26.7
3,058,Roma,RM,2024,121,3001253,86281665718,"29,710.0",33.5
1,016,Bergamo,BG,2024,243,844716,23052546583,"27,907.0",27.3
2,018,Pavia,PV,2024,185,411173,10926719711,"27,192.0",27.6
10,072,Bari,BA,2024,41,850680,18486827622,"22,297.0",41.0
7,063,Napoli,NA,2024,92,1677856,36098332358,"22,290.0",43.5
5,060,Frosinone,FR,2024,91,316439,6749267125,"21,992.0",37.7
4,059,Latina,LT,2024,33,399679,8494674865,"21,971.0",41.7
6,061,Caserta,CE,2024,104,544671,10818989173,"20,525.0",45.1
8,065,Salerno,SA,2024,158,696824,13862798057,"20,517.0",46.7


In [5]:
assert len(redditi) == 12, "Manca qualche provincia!"
assert redditi["reddito_medio"].notna().all(), "C'è un reddito medio mancante!"
print("Controlli superati ✅")

redditi.to_csv(CARTELLA_PULITI / "mef_redditi_province.csv", index=False)
print("Salvato in:", CARTELLA_PULITI / "mef_redditi_province.csv")

Controlli superati ✅
Salvato in: ..\dati_puliti\mef_redditi_province.csv
